# CAVE

The CAVE nodes, wired the way references exist for.

*Exported from Coda v0.0.0-test on 2026-01-01.*

In [ ]:
# pip install caveclient numpy pandas sea-serpent

import os
import pandas as pd
import numpy as np
from caveclient import CAVEclient
import seaserpent as ss

In [ ]:
# Helpers, generated by Coda. These are the parts of the workflow that have
# no equivalent in the libraries this notebook imports, written out here so
# it stands on its own.

def coda_int64(values):
    """A column of ids as exact int64. Anything unreadable becomes 0."""

    def one(value):
        try:
            # Exact for an int and for decimal text of any width, which pd.to_numeric is
            # not: a single null in the column makes it answer float64, and an
            # eighteen-digit root id through a float is a different neuron.
            return int(value)
        except (TypeError, ValueError):
            try:
                return int(float(value))
            except (TypeError, ValueError):
                return 0

    return values.map(one).astype('int64')


def coda_annotation_columns(df, id_column):
    """Rename a CAVE table's columns to the two names Coda addresses by name."""
    renames = {}
    if id_column in df.columns:
        renames[id_column] = 'neuronId'
    for name in ('cell_type', 'celltype'):
        if name in df.columns and 'type' not in df.columns:
            renames[name] = 'type'
            break
    out = df.rename(columns=renames)
    if 'neuronId' not in out.columns:
        return out
    # A row with no id names no neuron, which is what Coda's shaping drops. An empty
    # string counts: SeaTable spells a blank cell that way.
    ids = out['neuronId']
    out = out[ids.notna() & (ids.astype(str) != '')].copy()
    ids = out['neuronId']
    if pd.api.types.is_float_dtype(ids):
        # Already lossy — a float64 cannot hold an eighteen-digit root id — but `str()` of
        # one is `7.2e+17`, which matches nothing at all. Integer text at least keeps the
        # shape of an id. Text and int columns go straight through, exact at any width.
        ids = coda_int64(ids)
    out['neuronId'] = ids.astype(str)
    return out


def coda_cave_neurons(
    client,
    neuron_table,
    id_column='pt_root_id',
    annotation_table=None,
    ref_column=None,
    system_column=None,
    value_column=None,
):
    """Coda's neuron index for a CAVE datastack: one row per neuron, a column per kind."""
    neurons = client.materialize.query_table(
        neuron_table, select_columns=['id', id_column], merge_reference=False
    )
    if annotation_table is None:
        return coda_annotation_columns(neurons.drop(columns=['id']), id_column)

    kinds = client.materialize.get_unique_string_values(annotation_table).get(
        system_column, []
    )
    wide = None
    for kind in kinds:
        rows = client.materialize.query_table(
            annotation_table,
            filter_equal_dict={system_column: kind},
            select_columns=[ref_column, value_column],
            merge_reference=False,
        )
        # First row wins a repeat, as Coda's pivot does: an annotation base can carry two
        # rows for one neuron, and a cross product here would double every downstream count.
        rows = rows.drop_duplicates(subset=[ref_column], keep='first')
        rows = rows.rename(columns={value_column: kind})
        wide = rows if wide is None else wide.merge(rows, on=ref_column, how='outer')

    if wide is None:
        return coda_annotation_columns(neurons.drop(columns=['id']), id_column)
    out = neurons.merge(wide, left_on='id', right_on=ref_column, how='left')
    out = out.drop(columns=[c for c in ('id', ref_column) if c in out.columns])
    return coda_annotation_columns(out, id_column)


class CodaCaveDataset:
    """A CAVE datastack: the client, and the neuron table Coda labels it with.

    `client` is a `caveclient.CAVEclient` pinned to one materialization, so every query
    through it answers from the same frozen snapshot and `client.timestamp` is that
    snapshot's instant.

    `labels` is one row per neuron with Coda's column names — `neuronId`, `type` — built
    from the datastack's own tables, or handed over ready-made when something is wired to
    the Dataset's Annotations socket on the canvas. It is fetched on first use: a graph
    that never asks about neurons should not pay for the index.
    """

    def __init__(
        self,
        client,
        neuron_table=None,
        id_column='pt_root_id',
        annotation_table=None,
        ref_column=None,
        system_column=None,
        value_column=None,
        labels=None,
    ):
        self.client = client
        self.neuron_table = neuron_table
        self.id_column = id_column
        self.annotation_table = annotation_table
        self.ref_column = ref_column
        self.system_column = system_column
        self.value_column = value_column
        self._labels = labels

    @property
    def labels(self):
        if self._labels is None:
            if self.neuron_table is None:
                raise ValueError(
                    'This datastack publishes no neuron table, so the only list of its '
                    'neurons is an annotation source. Wire one to the Dataset on the canvas '
                    'and export again, or pass labels= here.'
                )
            self._labels = coda_cave_neurons(
                self.client,
                self.neuron_table,
                id_column=self.id_column,
                annotation_table=self.annotation_table,
                ref_column=self.ref_column,
                system_column=self.system_column,
                value_column=self.value_column,
            )
        return self._labels


def coda_cave_table(
    client,
    table,
    id_column='pt_root_id',
    columns=None,
    pivot_on=None,
    value_column=None,
):
    """A CAVE annotation table as a Coda neuron table."""
    if pivot_on:
        kinds = client.materialize.get_unique_string_values(table).get(pivot_on, [])
        wide = None
        for kind in kinds:
            rows = client.materialize.query_table(
                table,
                filter_equal_dict={pivot_on: kind},
                select_columns=[id_column, value_column],
                merge_reference=False,
            )
            rows = rows.drop_duplicates(subset=[id_column], keep='first')
            rows = rows.rename(columns={value_column: kind})
            wide = rows if wide is None else wide.merge(rows, on=id_column, how='outer')
        out = wide if wide is not None else pd.DataFrame({id_column: []})
    else:
        select = [id_column] + list(columns) if columns else None
        out = client.materialize.query_table(
            table, select_columns=select, merge_reference=False
        )
        if columns:
            out = out[[id_column] + [c for c in columns if c in out.columns]]
    return coda_annotation_columns(out, id_column)


def coda_join_annotations(left, right):
    """Chain two annotation sources: outer join on `neuronId`, the later one winning."""
    if left is None:
        return right
    if right is None:
        return left
    left = left.drop_duplicates(subset=['neuronId'], keep='first')
    right = right.drop_duplicates(subset=['neuronId'], keep='first')
    shared = [c for c in right.columns if c in left.columns and c != 'neuronId']
    merged = left.merge(
        right, on='neuronId', how='outer', suffixes=('', '_coda_later')
    )
    for name in shared:
        later = merged[name + '_coda_later']
        # Later wins, falling back to the earlier source where the later one is null.
        merged[name] = later.combine_first(merged[name])
        merged = merged.drop(columns=[name + '_coda_later'])
    return merged


def coda_seatable(table, id_column='root_id', columns=None):
    """A SeaTable table as a Coda neuron table."""
    df = table.to_frame(row_id_index=False)
    # sea-serpent names its columns with numpy `str_`, which indexes fine and reads oddly
    # in anything that prints the column list.
    df.columns = [str(c) for c in df.columns]
    if columns:
        keep = [c for c in columns if c in df.columns and c != id_column]
    else:
        # Empty means every column but the id, which is what a base says without being
        # asked what "every" is.
        keep = [c for c in df.columns if c != id_column]
    return coda_annotation_columns(df[[id_column] + keep], id_column)


def coda_update_root_ids(
    client, df, id_column='neuronId', supervoxel_column='supervoxel_id'
):
    """Repair root ids that were retired before this materialization was frozen."""
    out = df.copy()
    ids = coda_int64(out[id_column])
    svids = coda_int64(out[supervoxel_column])
    askable = ids > 0
    if not askable.any():
        return out

    # Current at the materialization? Only the rows that are not get looked up.
    latest = client.chunkedgraph.is_latest_roots(
        ids[askable].astype('int64').to_numpy(), timestamp=client.timestamp
    )
    stale = pd.Series(False, index=out.index)
    stale.loc[askable] = ~np.asarray(latest, dtype=bool)
    stale &= svids > 0
    if not stale.any():
        return out

    roots = client.chunkedgraph.get_roots(
        svids[stale].astype('int64').to_numpy(), timestamp=client.timestamp
    )
    repaired = pd.Series(np.asarray(roots), index=out.index[stale])
    # A supervoxel the graph does not know answers 0, which is not a root to write anywhere.
    repaired = repaired[repaired > 0]
    if repaired.empty:
        return out
    if out[id_column].dtype == object:
        repaired = repaired.astype(str)
    else:
        # Keep the column's own storage rather than widening it to uint64 or object,
        # which would change how every later comparison and sort behaves.
        repaired = repaired.astype(out[id_column].dtype)
    out.loc[repaired.index, id_column] = repaired
    return out

In [ ]:
# ── SeaTable ──
_sea = ss.Table(
    'types',
    base='my base',
    server='https://cloud.seatable.io',
    auth_token=os.environ['SEATABLE_TOKEN'],
)
seatable = coda_seatable(
    _sea,
    id_column='root_id',
)

In [ ]:
# ── Custom CAVE ──
custom_cave = CodaCaveDataset(
    CAVEclient('wclee_aedes_brain', version=117),
    neuron_table='nuclei',
    id_column='pt_root_id',
)

In [ ]:
# ── FlyTable ──
_sea = ss.Table(
    'info',
    base='main',
    server='https://flytable.mrc-lmb.cam.ac.uk',
    auth_token=os.environ['SEATABLE_TOKEN'],
)
# Every column is downloaded and then narrowed, as it is on the canvas. To narrow it
# server-side instead — measured at about 4x faster, at the cost of sea-serpent's
# dtype conversion — replace the call below with:
#     _rows = _sea.query('SELECT `root_id`, `cell_type`, `side` FROM `info`', no_limit=True)
#     flytable = coda_annotation_columns(pd.DataFrame(_rows), 'root_id')
flytable = coda_seatable(
    _sea,
    id_column='root_id',
    columns=['cell_type', 'side'],
)
flytable = coda_join_annotations(seatable, flytable)

In [ ]:
# ── CAVE table ──
_cave = CAVEclient('flywire_fafb_public', version=783)
cave_table = coda_cave_table(
    _cave,
    'nuclei_v1',
    id_column='pt_root_id',
    columns=['volume'],
)
cave_table = coda_join_annotations(flytable, cave_table)

In [ ]:
# ── CAVE table ──
# NOTE: The Dataset wired here is a reference — it names a datastack rather
# than taking its value — and its cell is written below this one, so this
# builds its own client for the same datastack and materialization.
_cave = CAVEclient('flywire_fafb_public', version=783)
cave_table_2 = coda_cave_table(
    _cave,
    'hierarchical_neuron_annotations',
    id_column='target_id',
    pivot_on='classification_system',
    value_column='cell_type',
)
cave_table_2 = coda_join_annotations(cave_table, cave_table_2)

In [ ]:
# ── Filter ──
filter_ = cave_table_2[cave_table_2['type'].notna() & (cave_table_2['type'] != '')]

In [ ]:
# ── Update root IDs ──
# NOTE: The Dataset wired here is a reference — it names a datastack rather
# than taking its value — and its cell is written below this one, so this
# builds its own client for the same datastack and materialization.
_repair_at = CAVEclient('flywire_fafb_public', version=783)
update_root_ids = coda_update_root_ids(
    _repair_at,
    filter_,
    id_column='neuronId',
    supervoxel_column='supervoxel_id',
)

In [ ]:
# ── FlyWire FAFB (CAVE) ──
# NOTE: Annotations are wired to this dataset on the canvas, so they replace
# the datastack's own labels rather than adding to them.
flywire_fafb_cave = CodaCaveDataset(
    CAVEclient('flywire_fafb_public', version=783),
    labels=update_root_ids,
)

In [ ]:
# ── Table ──
table_out = update_root_ids
table_filtered = table_out
table_out

In [ ]:
# ── Find Neurons ──
# TODO: "Find Neurons" is wired to a CAVE dataset, and its notebook cell has
# only been written for neuPrint. The dataset itself is a real client, so
# this is the step to fill in by hand.